# Read interpolated light curves

In [8]:
import numpy as np
import polars as pl
from sklearn.manifold import TSNE

from snanomaly import dirs
import snanomaly.models.results.util as resutil

In [9]:
# get input
dataset_name = "osc2018_june"
df = pl.read_parquet(dirs.INTERPOLATED / f"{dataset_name}_LSB-STATIC" / f"{dataset_name}.parquet")
df

sn_name,bandset,peak_time,days_pre_peak,days_post_peak,log_likelihood,thetas,pred_means,pred_stds
str,list[str],f64,i64,i64,f64,list[f64],list[list[f64]],list[list[f64]]
"""SDSS-II SN 17907""","[""g_pr"", ""r_pr"", ""i_pr""]",54360.5,20,100,40.859446,"[1.817474, 0.976793, … -0.112782]","[[1.0836e-9, 1.7216e-9, … 0.0], [1.5718e-9, 2.4974e-9, … 0.0], [1.5095e-9, 2.3984e-9, … 0.0]]","[[3.1032e-8, 3.1026e-8, … 0.0], [4.7504e-8, 4.7496e-8, … 0.0], [4.7784e-8, 4.7776e-8, … 0.0]]"
"""SDSS-II SN 15074""","[""g_pr"", ""r_pr"", ""i_pr""]",54019.5,20,100,20.305697,"[1.040616, 0.006335, … -0.185931]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0]]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0]]"
"""PS1-12bku""","[""g_pr"", ""r_pr"", ""i_pr""]",56206.5,20,100,5.712248,"[1.485435, 1.028905, … -0.212365]","[[3.1984e-9, 4.0253e-9, … 5.2086e-16], [4.3131e-9, 5.2261e-9, … 7.6584e-16], [5.2259e-9, 6.4279e-9, … 9.0266e-16]]","[[3.0114e-9, 2.6499e-9, … 3.5441e-9], [4.6124e-9, 4.0861e-9, … 5.3755e-9], [5.7363e-9, 5.1584e-9, … 6.5915e-9]]"
"""PS1-12bku""","[""g"", ""r"", ""i""]",56174.5,20,100,9.002704,"[0.378619, 0.112686, … -4.6510e-9]","[[1.1954e-14, 2.5198e-13, … 3.3133e-20], [1.8651e-14, 3.9316e-13, … 5.1691e-20], [2.8754e-14, 6.0613e-13, … 7.9693e-20]]","[[2.8726e-9, 2.8726e-9, … 2.8726e-9], [4.5551e-9, 4.5551e-9, … 4.5551e-9], [7.0226e-9, 7.0226e-9, … 7.0226e-9]]"
"""SDSS-II SN 20592""","[""g_pr"", ""r_pr"", ""i_pr""]",54405.5,20,100,26.328436,"[1.245873, 1.124316, … 0.192283]","[[1.3023e-8, 1.1623e-8, … 0.0], [2.2930e-8, 1.9697e-8, … 2.7915e-126], [2.4090e-8, 1.9549e-8, … 4.2657e-126]]","[[4.7809e-13, 1.8088e-10, … 0.0], [4.7809e-13, 1.7267e-9, … 1.4106e-8], [4.7809e-13, 3.8439e-9, … 2.1556e-8]]"
…,…,…,…,…,…,…,…,…
"""SN2016ayf""","[""g"", ""r"", ""i""]",57464.5,20,100,9.115077,"[0.00002, 0.000018, … 0.054203]","[[6.7301e-93, 1.9789e-84, … 0.0], [7.4106e-93, 2.1790e-84, … 0.0], [4.0208e-93, 1.1823e-84, … 0.0]]","[[0.000002, 0.000002, … 0.0], [0.000003, 0.000003, … 0.0], [0.000002, 0.000002, … 0.0]]"
"""SN2007ai""","[""g"", ""r"", ""i""]",54176.5,20,100,61.49233,"[1.616266, 1.287578, … 0.042188]","[[6.4079e-10, 1.1244e-9, … 7.3917e-12], [1.0794e-9, 1.8976e-9, … 1.2469e-11], [8.0077e-10, 1.4090e-9, … 9.2568e-12]]","[[2.1474e-8, 2.1465e-8, … 2.1478e-8], [3.7877e-8, 3.7862e-8, … 3.7884e-8], [2.9635e-8, 2.9624e-8, … 2.9640e-8]]"
"""PTF11dec""","[""g"", ""r"", ""i""]",55704.5,20,100,35.527299,"[2.277329, 1.5342, … 0.059556]","[[2.8156e-9, 3.3589e-9, … 0.0], [4.1483e-9, 4.9497e-9, … 0.0], [2.6185e-9, 3.1252e-9, … 0.0]]","[[7.3147e-9, 7.1975e-9, … 0.0], [1.0879e-8, 1.0708e-8, … 0.0], [7.2381e-9, 7.1353e-9, … 0.0]]"


### Prepare data for learning algorithms

In [10]:
# select columns of interest
light_curves = df.select(["sn_name", "bandset", "pred_means", "log_likelihood", "thetas"])

In [11]:
# normalize, standardize
light_curves = light_curves.with_columns(
    # normalize prediction mean values per candidate by the maximum of flux maximums
    pl.col("pred_means").map_elements(lambda x: x / max((xx.max() for xx in x)), return_dtype=list[list[float]]),
    # normalize theta values per candidate by the difference between maximum and minimum values
    pl.col("thetas").map_elements(lambda x: x / (x.max() - x.min()), return_dtype=list[float]),
    # standardize log-likelihood across all candidates
    (pl.col("log_likelihood") - pl.col("log_likelihood").mean()) / pl.col("log_likelihood").std(),
)
light_curves
# TODO: transform flux values to `gri` if band set is not `gri`

sn_name,bandset,pred_means,log_likelihood,thetas
str,list[str],list[list[f64]],f64,list[f64]
"""SDSS-II SN 17907""","[""g_pr"", ""r_pr"", ""i_pr""]","[[0.008535, 0.013561, … 0.0], [0.012381, 0.019671, … 0.0], [0.01189, 0.018891, … 0.0]]",-0.068946,"[0.941572, 0.506043, … -0.058428]"
"""SDSS-II SN 15074""","[""g_pr"", ""r_pr"", ""i_pr""]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0]]",-0.404479,"[0.81677, 0.004972, … -0.145935]"
"""PS1-12bku""","[""g_pr"", ""r_pr"", ""i_pr""]","[[0.307964, 0.387573, … 5.0152e-8], [0.41529, 0.503201, … 7.3739e-8], [0.503183, 0.618912, … 8.6913e-8]]",-0.642712,"[0.874917, 0.606022, … -0.125083]"
"""PS1-12bku""","[""g"", ""r"", ""i""]","[[0.000001, 0.000025, … 3.2528e-12], [0.000002, 0.000039, … 5.0748e-12], [0.000003, 0.00006, … 7.8238e-12]]",-0.588997,"[0.558184, 0.166129, … -6.8568e-9]"
"""SDSS-II SN 20592""","[""g_pr"", ""r_pr"", ""i_pr""]","[[0.26558, 0.237046, … 0.0], [0.467633, 0.401699, … 5.6929e-119], [0.491298, 0.398685, … 8.6995e-119]]",-0.30616,"[1.182503, 1.067128, … 0.182503]"
…,…,…,…,…
"""SN2016ayf""","[""g"", ""r"", ""i""]","[[1.2671e-87, 3.7256e-79, … 0.0], [1.3952e-87, 4.1023e-79, … 0.0], [7.5698e-88, 2.2259e-79, … 0.0]]",-0.587162,"[0.000036, 0.000032, … 0.098048]"
"""SN2007ai""","[""g"", ""r"", ""i""]","[[0.006907, 0.01212, … 0.00008], [0.011634, 0.020454, … 0.000134], [0.008631, 0.015187, … 0.0001]]",0.267879,"[1.026802, 0.817989, … 0.026802]"
"""PTF11dec""","[""g"", ""r"", ""i""]","[[0.102895, 0.122751, … 0.0], [0.1516, 0.180887, … 0.0], [0.095693, 0.114208, … 0.0]]",-0.155991,"[1.026854, 0.691775, … 0.026854]"


In [12]:
# converting list elements to columns
light_curves = resutil.explode_lists_to_numbered_columns(light_curves, "pred_means")
light_curves = resutil.explode_lists_to_numbered_columns(light_curves, "thetas")
light_curves

sn_name,bandset,log_likelihood,pred_means_0,pred_means_1,pred_means_2,pred_means_3,pred_means_4,pred_means_5,pred_means_6,pred_means_7,pred_means_8,pred_means_9,pred_means_10,pred_means_11,pred_means_12,pred_means_13,pred_means_14,pred_means_15,pred_means_16,pred_means_17,pred_means_18,pred_means_19,pred_means_20,pred_means_21,pred_means_22,pred_means_23,pred_means_24,pred_means_25,pred_means_26,pred_means_27,pred_means_28,pred_means_29,pred_means_30,pred_means_31,pred_means_32,pred_means_33,…,pred_means_335,pred_means_336,pred_means_337,pred_means_338,pred_means_339,pred_means_340,pred_means_341,pred_means_342,pred_means_343,pred_means_344,pred_means_345,pred_means_346,pred_means_347,pred_means_348,pred_means_349,pred_means_350,pred_means_351,pred_means_352,pred_means_353,pred_means_354,pred_means_355,pred_means_356,pred_means_357,pred_means_358,pred_means_359,pred_means_360,pred_means_361,pred_means_362,thetas_0,thetas_1,thetas_2,thetas_3,thetas_4,thetas_5,thetas_6,thetas_7,thetas_8
str,list[str],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""SDSS-II SN 17907""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.068946,0.008535,0.013561,0.02099,0.031652,0.046505,0.066577,0.092879,0.126275,0.167332,0.216154,0.272235,0.334353,0.400539,0.468143,0.533992,0.594652,0.646734,0.687231,0.71381,0.725029,0.720445,0.700601,0.6669,0.62141,0.566637,0.505311,0.440217,0.374096,0.309606,0.249311,0.195663,0.150934,0.117052,0.095341,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.941572,0.506043,0.112341,0.130362,0.189101,0.063746,0.181604,0.062441,-0.058428
"""SDSS-II SN 15074""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.404479,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005196,0.138985,0.271331,0.348563,0.356462,0.324635,0.296797,0.297661,0.325306,0.366267,0.409645,0.447123,0.467434,0.458477,0.417196,0.35605,0.298162,0.264704,0.264215,0.289744,0.322652,0.339636,0.321634,0.263408,0.17903,0.097113,0.045665,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.81677,0.004972,0.358616,0.171246,0.310953,-0.147243,0.342232,-0.18323,-0.145935
"""PS1-12bku""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.642712,0.307964,0.387573,0.463108,0.525143,0.564648,0.574858,0.552816,0.500163,0.423049,0.33129,0.237068,0.15342,0.092612,0.064332,0.073746,0.119757,0.194136,0.282293,0.366067,0.428115,0.45662,0.448621,0.410618,0.356091,0.300856,0.25802,0.234369,0.229307,0.236367,0.246355,0.25074,0.244114,0.225118,0.195922,…,0.236215,0.310517,0.377384,0.405099,0.377384,0.310517,0.236215,0.17455,0.128183,0.09284,0.065222,0.043951,0.028269,0.017319,0.010095,0.005595,0.002947,0.001475,0.000701,0.000317,0.000136,0.000055,0.000021,0.000008,0.000003,9.1856e-7,2.8989e-7,8.6913e-8,0.874917,0.606022,0.368907,0.208747,0.306924,-0.077728,0.361759,-0.064894,-0.125083
"""PS1-12bku""","[""g"", ""r"", ""i""]",-0.588997,0.000001,0.000025,0.000326,0.002693,0.013913,0.045119,0.0927,0.125449,0.128834,0.138814,0.187068,0.249622,0.261537,0.190859,0.091469,0.029068,0.01483,0.049426,0.15758,0.318324,0.402439,0.318322,0.157531,0.048775,0.009449,0.001145,0.000087,0.000004,1.2228e-7,2.2709e-9,2.6385e-11,1.9180e-13,8.7235e-16,2.4823e-18,…,0.001407,0.000107,0.000005,1.5015e-7,2.7884e-9,3.2398e-11,2.3551e-13,1.0711e-15,3.0480e-18,5.4263e-21,6.0442e-24,4.2121e-27,1.8365e-30,5.0099e-34,8.5506e-38,9.1304e-42,7.5639e-46,2.1914e-42,2.0523e-38,1.2025e-34,4.4087e-31,1.0113e-27,1.4515e-24,1.3036e-21,7.3262e-19,2.5770e-16,5.6743e-14,7.8238e-12,0.558184,0.166129,0.117584,0.415761,0.648634,0.118018,1.0,0.181948,-6.8568e-9
"""SDSS-II SN 20592""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.30616,0.26558,0.237046,0.205412,0.188301,0.191578,0.207569,0.22221,0.225205,0.216283,0.204652,0.203672

# t-SNE

In [13]:
def square_distance(X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
    return np.sum((X1 - X2) ** 2)

tsne = TSNE(n_components=)

SyntaxError: expected argument value expression (2145521979.py, line 4)

# UMAP

In [ ]:
num_obs = len(df.select(["pred_means"][0])[0])
num_obs